# Sui Unlock Recipient Explorer

Quickly inspect the largest SUI recipients around a specific unlock. This configuration focuses on the May 2, 2024 unlock and scans one hour before and after the event using the public JSON-RPC endpoint.


Parameters in the next cells let you tune how far before/after each unlock to search and how many addresses to surface. The implementation avoids non-standard dependencies aside from `requests`; if `pandas` is available it will be used for display, otherwise plain Python output is shown.


In [54]:
import math
import time
import csv
import re
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
from collections import defaultdict
from functools import lru_cache
from pathlib import Path

import requests

try:
    from IPython.display import display
except ImportError:
    display = print

MIST_PER_SUI = 10 ** 9
RPC_URL = "https://fullnode.mainnet.sui.io:443"
REQUEST_TIMEOUT = 10
MAX_RETRIES = 4

def resolve_data_dir() -> Path:
    search_roots = [Path.cwd()] + list(Path.cwd().parents)
    for base in search_roots:
        if base.name == 'scripts':
            candidate = base.parent / 'data'
            return candidate.resolve()
        if base.name == 'sui_insiders':
            return (base / 'data').resolve()
        nested = base / 'sui_insiders'
        if nested.exists():
            return (nested / 'data').resolve()
        sibling_data = base / 'data'
        sibling_scripts = base / 'scripts'
        if sibling_data.exists() and sibling_scripts.exists():
            return sibling_data.resolve()
    return (Path.cwd() / 'data').resolve()

DATA_DIR = resolve_data_dir()
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [55]:
LOCAL_ZONE = ZoneInfo("Asia/Singapore")

@dataclass
class UnlockEvent:
    label: str
    local_time: datetime
    utc_time: datetime
    timestamp_ms: int

def make_event(label: str, year: int, month: int, day: int, hour: int, minute: int) -> UnlockEvent:
    local_dt = datetime(year, month, day, hour, minute, tzinfo=LOCAL_ZONE)
    utc_dt = local_dt.astimezone(timezone.utc)
    timestamp_ms = int(utc_dt.timestamp() * 1000)
    return UnlockEvent(label=label, local_time=local_dt, utc_time=utc_dt, timestamp_ms=timestamp_ms)

EVENTS: List[UnlockEvent] = [
    # make_event("2024-04-02 Unlock", 2024, 4, 2, 3, 0),
    # make_event("2024-05-02 Unlock", 2024, 5, 2, 13, 30),
    # make_event("2024-06-02 Unlock", 2024, 6, 2, 0, 0),
    # make_event("2024-07-02 Unlock", 2024, 7, 2, 10, 30),
    # make_event("2024-08-01 Unlock", 2024, 8, 1, 21, 0),
    # make_event("2024-09-01 Unlock", 2024, 9, 1, 7, 30),
    make_event("2024-10-01 Unlock", 2024, 10, 1, 18, 0),
]

try:
    import pandas as pd  # type: ignore
except ImportError:
    pd = None

event_rows = [
    {
        "Event": event.label,
        "Local (GMT+8)": event.local_time.isoformat(),
        "UTC": event.utc_time.isoformat(),
        "Timestamp (ms)": event.timestamp_ms,
    }
    for event in EVENTS
]

if pd is not None:
    display(pd.DataFrame(event_rows))
else:
    for row in event_rows:
        print(row)


,Event,Local (GMT+8),UTC,Timestamp (ms)
0,2024-10-01 Unlock,2024-10-01T18:00:00+08:00,2024-10-01T10:00:00+00:00,1727776800000


In [56]:
def rpc_request(method: str, params: Optional[Sequence[Any]] = None) -> Any:
    payload = {
        "jsonrpc": "2.0",
        "id": int(time.time() * 1000) % 1_000_000,
        "method": method,
        "params": list(params) if params is not None else [],
    }
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(RPC_URL, json=payload, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            body = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(1.5 ** attempt)
            continue
        if "error" in body:
            raise RuntimeError(f"RPC error for {method}: {body['error']}")
        return body.get("result")
    raise RuntimeError(f"RPC request failed after retries: {method}") from last_error

def batched(values: Sequence[Any], size: int) -> Iterable[Sequence[Any]]:
    for offset in range(0, len(values), size):
        yield values[offset : offset + size]

def get_latest_checkpoint() -> int:
    latest = rpc_request("sui_getLatestCheckpointSequenceNumber")
    return int(latest)

@lru_cache(maxsize=None)
def get_checkpoint(sequence_number: int) -> Dict[str, Any]:
    result = rpc_request("sui_getCheckpoint", [str(sequence_number)])
    if result is None:
        raise RuntimeError(f"Missing checkpoint data for {sequence_number}")
    return result

def checkpoint_timestamp(sequence_number: int) -> Optional[int]:
    data = get_checkpoint(sequence_number)
    ts = data.get("timestampMs")
    return int(ts) if ts is not None else None

def find_checkpoint_at_or_after(target_timestamp_ms: int) -> Optional[int]:
    low = 0
    high = get_latest_checkpoint()
    candidate: Optional[int] = None
    while low <= high:
        mid = (low + high) // 2
        mid_ts = checkpoint_timestamp(mid)
        if mid_ts is None:
            high = mid - 1
            continue
        if mid_ts < target_timestamp_ms:
            low = mid + 1
        else:
            candidate = mid
            high = mid - 1
    return candidate

def find_checkpoint_at_or_before(target_timestamp_ms: int) -> Optional[int]:
    low = 0
    high = get_latest_checkpoint()
    candidate: Optional[int] = None
    while low <= high:
        mid = (low + high) // 2
        mid_ts = checkpoint_timestamp(mid)
        if mid_ts is None or mid_ts > target_timestamp_ms:
            high = mid - 1
        else:
            candidate = mid
            low = mid + 1
    return candidate


In [57]:
def load_checkpoints_between(start_sequence: int, end_sequence: int, *, batch_size: int = 100) -> List[Dict[str, Any]]:
    if end_sequence < start_sequence:
        return []
    checkpoints: List[Dict[str, Any]] = []
    cursor: Optional[str] = str(start_sequence - 1) if start_sequence > 0 else None
    remaining = end_sequence - start_sequence + 1
    while len(checkpoints) < remaining:
        limit = min(batch_size, remaining - len(checkpoints))
        response = rpc_request("sui_getCheckpoints", [cursor, limit, False])
        data = response.get("data", []) if isinstance(response, dict) else []
        if not data:
            break
        for checkpoint in data:
            seq_raw = checkpoint.get("sequenceNumber")
            if seq_raw is None:
                continue
            seq = int(seq_raw)
            if seq < start_sequence or seq > end_sequence:
                continue
            checkpoints.append(checkpoint)
        cursor = data[-1].get("sequenceNumber")
        if len(data) < limit:
            break
    checkpoints.sort(key=lambda item: int(item.get("sequenceNumber", 0)))
    return checkpoints


In [58]:
def load_transactions_for_checkpoints(checkpoints: Sequence[Dict[str, Any]], *, batch_size: int = 50, max_transactions: Optional[int] = None) -> Tuple[List[Dict[str, Any]], bool]:
    digests: List[str] = []
    for checkpoint in checkpoints:
        digests.extend(checkpoint.get("transactions", []))
    truncated = False
    if max_transactions is not None and len(digests) > max_transactions:
        digests = digests[:max_transactions]
        truncated = True
    transactions: List[Dict[str, Any]] = []
    if not digests:
        return transactions, truncated
    options = {
        "showBalanceChanges": True,
        "showInput": True,
    }
    for chunk in batched(digests, batch_size):
        result = rpc_request("sui_multiGetTransactionBlocks", [list(chunk), options])
        transactions.extend(result)
    return transactions, truncated


In [59]:
def summarize_sui_transfers(transactions: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    stats: Dict[str, Dict[str, Any]] = {}
    for tx in transactions:
        digest = tx.get("digest")
        for change in tx.get("balanceChanges") or []:
            if change.get("coinType") != "0x2::sui::SUI":
                continue
            owner_info = change.get("owner") or {}
            owner = owner_info.get("AddressOwner")
            if owner is None:
                continue
            amount = int(change.get("amount", 0))
            entry = stats.setdefault(owner, {
                "owner": owner,
                "received_mist": 0,
                "sent_mist": 0,
                "net_mist": 0,
                "tx_digests": set(),
            })
            if amount > 0:
                entry["received_mist"] += amount
            elif amount < 0:
                entry["sent_mist"] += -amount
            entry["net_mist"] += amount
            if digest:
                entry["tx_digests"].add(digest)
    rows: List[Dict[str, Any]] = []
    for entry in stats.values():
        rows.append({
            "owner": entry["owner"],
            "received_sui": entry["received_mist"] / MIST_PER_SUI,
            "sent_sui": entry["sent_mist"] / MIST_PER_SUI,
            "net_sui": entry["net_mist"] / MIST_PER_SUI,
            "tx_count": len(entry["tx_digests"]),
        })
    rows.sort(key=lambda row: row["net_sui"], reverse=True)
    return rows


In [60]:
def analyze_event(event: UnlockEvent, *, window_before_minutes: int, window_after_minutes: int, batch_size: int = 50, max_transactions: Optional[int] = None) -> Dict[str, Any]:
    window_start_ms = event.timestamp_ms - window_before_minutes * 60 * 1000
    window_end_ms = event.timestamp_ms + window_after_minutes * 60 * 1000
    start_checkpoint = find_checkpoint_at_or_after(window_start_ms)
    end_checkpoint = find_checkpoint_at_or_before(window_end_ms)
    latest_checkpoint = get_latest_checkpoint()
    if start_checkpoint is None:
        start_checkpoint = 0
    if end_checkpoint is None or end_checkpoint > latest_checkpoint:
        end_checkpoint = latest_checkpoint
    if start_checkpoint > end_checkpoint:
        return {
            "event": event,
            "window": (window_start_ms, window_end_ms),
            "start_checkpoint": start_checkpoint,
            "end_checkpoint": end_checkpoint,
            "num_checkpoints": 0,
            "num_transactions": 0,
            "num_transactions_fetched": 0,
            "truncated": False,
            "top_addresses": [],
            "transactions": [],
        }
    checkpoints = load_checkpoints_between(start_checkpoint, end_checkpoint)
    transactions, truncated = load_transactions_for_checkpoints(checkpoints, batch_size=batch_size, max_transactions=max_transactions)
    summary_rows = summarize_sui_transfers(transactions)
    total_transactions = sum(len(cp.get("transactions", [])) for cp in checkpoints)
    return {
        "event": event,
        "window": (window_start_ms, window_end_ms),
        "start_checkpoint": start_checkpoint,
        "end_checkpoint": end_checkpoint,
        "num_checkpoints": len(checkpoints),
        "num_transactions": total_transactions,
        "num_transactions_fetched": len(transactions),
        "truncated": truncated,
        "top_addresses": summary_rows,
        "transactions": transactions,
    }


In [61]:
def sanitize_label(label: str) -> str:
    slug = re.sub(r'[^0-9a-zA-Z]+', '_', label.strip().lower())
    slug = re.sub(r'_+', '_', slug).strip('_')
    return slug or 'event'

def balance_change_rows(result: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    event = result["event"]
    for tx in result.get("transactions", []):
        digest = tx.get("digest")
        checkpoint = tx.get("checkpoint")
        timestamp_ms = tx.get("timestampMs")
        tx_data = (tx.get("transaction") or {}).get("data") or {}
        tx_sender = tx_data.get("sender")
        for change in tx.get("balanceChanges") or []:
            if change.get("coinType") != "0x2::sui::SUI":
                continue
            amount_mist = int(change.get("amount", 0))
            if amount_mist <= 0:
                continue
            owner_info = change.get("owner") or {}
            recipient = owner_info.get("AddressOwner")
            if recipient is None:
                continue
            rows.append({
                'event_label': event.label,
                'tx_digest': digest,
                'checkpoint': checkpoint,
                'timestamp_ms': timestamp_ms,
                'sender_address': tx_sender,
                'recipient_address': recipient,
                'amount_mist': amount_mist,
                'amount_sui': amount_mist / MIST_PER_SUI,
            })
    return rows

def write_event_csv(result: Dict[str, Any]) -> tuple[Optional[Path], int]:
    rows = balance_change_rows(result)
    if not rows:
        return None, 0
    label = result['event'].label
    filename = sanitize_label(label) + '.csv'
    filepath = (DATA_DIR / filename).resolve()
    fieldnames = ['event_label', 'tx_digest', 'checkpoint', 'timestamp_ms', 'sender_address', 'recipient_address', 'amount_mist', 'amount_sui']
    with filepath.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return filepath, len(rows)


In [62]:
# Tunable parameters for the focused unlock check.
WINDOW_BEFORE_MINUTES = 10   # one hour before the unlock
WINDOW_AFTER_MINUTES = 10    # one hour after the unlock
TOP_N_ADDRESSES = 10
MAX_TRANSACTIONS = None     # cap fetch size to keep runtime reasonable
BATCH_SIZE = 50              # RPC batch size for multi-get


In [63]:
results: List[Dict[str, Any]] = []
for event in EVENTS:
    print(f"Analyzing {event.label} (window: -{WINDOW_BEFORE_MINUTES}m / +{WINDOW_AFTER_MINUTES}m)...")
    result = analyze_event(
        event,
        window_before_minutes=WINDOW_BEFORE_MINUTES,
        window_after_minutes=WINDOW_AFTER_MINUTES,
        batch_size=BATCH_SIZE,
        max_transactions=MAX_TRANSACTIONS,
    )
    print("  checkpoints: {num_checkpoints}, transactions collected: {num_transactions}, transactions fetched: {num_transactions_fetched}".format(**result))
    if result["truncated"]:
        print("  NOTE: transactions truncated to MAX_TRANSACTIONS")
    filepath, row_count = write_event_csv(result)
    if row_count:
        print(f"  wrote {row_count} recipient rows to {filepath}")
    else:
        print("  No SUI inflows captured; skipped CSV export.")
    results.append(result)
print("Analysis complete.")


Analyzing 2024-10-01 Unlock (window: -10m / +10m)...
  checkpoints: 4940, transactions collected: 57781, transactions fetched: 57781
  wrote 15690 recipient rows to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/2024_10_01_unlock.csv
Analysis complete.


In [ ]:
def display_event_result(result: Dict[str, Any], *, top_n: int) -> None:
    event = result["event"]
    window_start_ms, window_end_ms = result["window"]
    window_start = datetime.fromtimestamp(window_start_ms / 1000, tz=timezone.utc)
    window_end = datetime.fromtimestamp(window_end_ms / 1000, tz=timezone.utc)
    header_lines = [
        f"=== {event.label} ===",
        f"Unlock (local GMT+8): {event.local_time.isoformat()}",
        f"Window (UTC): {window_start.isoformat()} to {window_end.isoformat()}",
        f"Checkpoints: {result['start_checkpoint']} -> {result['end_checkpoint']}",
        f"Transactions fetched: {result['num_transactions_fetched']} of {result['num_transactions']} total in window",
    ]
    print("\n".join(header_lines))
    rows = result.get("top_addresses", [])[:top_n]
    if not rows:
        print("No SUI balance changes captured in this window.")
        return
    if pd is not None:
        df = pd.DataFrame(rows[:top_n])
        display(df)
    else:
        for row in rows:
            print(row)
    if result.get("truncated"):
        print("WARNING: Results truncated by MAX_TRANSACTIONS. Increase the limit to see more.")
    print()

for result in results:
    display_event_result(result, top_n=TOP_N_ADDRESSES)


In [ ]:
# Aggregate addresses across events to highlight repeat recipients.
address_aggregate: Dict[str, Dict[str, Any]] = {}
for result in results:
    event = result["event"]
    for row in result.get("top_addresses", []):
        owner = row["owner"]
        entry = address_aggregate.setdefault(owner, {
            "owner": owner,
            "total_net_sui": 0.0,
            "total_received_sui": 0.0,
            "events": [],
        })
        entry["total_net_sui"] += row["net_sui"]
        entry["total_received_sui"] += row["received_sui"]
        entry["events"].append({"label": event.label, "net_sui": row["net_sui"]})

aggregate_rows = [
    {
        "owner": data["owner"],
        "events_count": len(data["events"]),
        "total_net_sui": data["total_net_sui"],
        "total_received_sui": data["total_received_sui"],
    }
    for data in address_aggregate.values()
]
aggregate_rows.sort(key=lambda row: (row["events_count"], row["total_net_sui"]), reverse=True)
if pd is not None:
    display(pd.DataFrame(aggregate_rows))
else:
    for row in aggregate_rows:
        print(row)
